## Generate superpixel-based pseudolabels


### Overview

This is the third step for data preparation

Input: normalized images

Output: pseulabel label candidates for all the images

In [1]:
%reset
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import copy
import skimage

from skimage.segmentation import slic
from skimage.segmentation import mark_boundaries
from skimage.util import img_as_float
from skimage.measure import label 
import scipy.ndimage.morphology as snm
from skimage import io
import argparse
import numpy as np
import glob

import SimpleITK as sitk
import os

to01 = lambda x: (x - x.min()) / (x.max() - x.min())



Once deleted, variables cannot be recovered. Proceed (y/[n])? y


/home/cvpr/anaconda3/lib/python3.7/site-packages/skimage/io/manage_plugins.py:23: UserWarning: Your installed pillow version is < 8.1.2. Several security issues (CVE-2021-27921, CVE-2021-25290, CVE-2021-25291, CVE-2021-25293, and more) have been fixed in pillow 8.1.2 or higher. We recommend to upgrade this library.
  from .collection import imread_collection_wrapper


In [2]:
import cv2

**Summary**

a. Generate a mask of the patient to avoid pseudolabels of empty regions in the background

b. Generate superpixels as pseudolabels

**Configurations of pseudlabels**

```python
# default setting of minimum superpixel sizes
segs = seg_func(img[ii, ...], min_size = 400, sigma = 1)
# you can also try other configs
segs = seg_func(img[ii, ...], min_size = 100, sigma = 0.8)
```


In [3]:
DATASET_CONFIG = {'SABS':{
                    'img_bname': f'/media/cvpr/4231E076490AAA38/Aditya/datasets/SABS/Cervix/RawData/Training/sabs_CT_normalized/image_*.nii.gz',
                    'out_dir': './media/cvpr/4231E076490AAA38/Aditya/datasets/SABS/Cervix/RawData/Training/sabs_CT_normalized',
                    'fg_thresh': 1e-4
                    },
                  'CHAOST2':{
                      'img_bname': f'/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_*.nii.gz',
                      'out_dir': '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized',
                      'fg_thresh': 1e-4 + 50
                    }
                 }
            




In [4]:
DOMAIN = 'CHAOST2'
img_bname = DATASET_CONFIG[DOMAIN]['img_bname']
imgs = glob.glob(img_bname)
out_dir = DATASET_CONFIG[DOMAIN]['out_dir']

In [5]:
imgs

['/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_5.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_8.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_1.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_39.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_10.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_13.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_15.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_19.nii.gz',
 '/media/cvpr/4231E076490AA

In [6]:
imgs = sorted(imgs, key = lambda x: int(x.split('_')[-1].split('.nii.gz')[0]) )

In [7]:
imgs

['/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_1.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_2.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_3.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_5.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_8.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_10.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_13.nii.gz',
 '/media/cvpr/4231E076490AAA38/Aditya/datasets/CHAOS/CHAOS_Train_Sets/Train_Sets/chaos_MR_T2_normalized/image_15.nii.gz',
 '/media/cvpr/4231E076490AAA3

In [8]:
import networkx as nx
from skimage import segmentation, color
from skimage.future import graph


class edge:
    def __init__(self, x,y):
        self.x = x
        self.y = y

def get_bg_label(g):
    g_ = g.copy()
    bg_labels = []
    comps = nx.connected_components(g_)
    for i, nodes in enumerate(comps):
        for node in g_.nodes:
            if g_.nodes[node]['mean color'][0] < DATASET_CONFIG[DOMAIN]['fg_thresh']:
                for label in g_.nodes[node]['labels']:
                    bg_labels.append(label)
    return bg_labels

def cut_thresh(g, labels1, thresh):
    # Because deleting edges while iterating through them produces an error.
    g_ = g.copy()
    bg_labels = get_bg_label(g_)
    to_remove = []
    to_merge = []
    # print('here')
    for x, y, d in g_.edges(data=True):
        # print(x,y)
        color_diff = abs(g_.nodes[x]['mean color'][0] - g_.nodes[y]['mean color'][0])
        if_color_diff = color_diff > thresh
        is_x_bg = g_.nodes[x]['mean color'][0] < DATASET_CONFIG[DOMAIN]['fg_thresh']
        is_y_bg = g_.nodes[y]['mean color'][0] < DATASET_CONFIG[DOMAIN]['fg_thresh']
        is_both_bg = is_x_bg and is_y_bg
        if (is_x_bg or is_y_bg) and not is_both_bg:
            to_remove.append((x,y))
        elif if_color_diff and not is_both_bg:
            to_remove.append((x, y))
        elif not if_color_diff and (not is_x_bg and not is_y_bg) and (x,y) not in to_remove:
            to_merge.append(edge(x,y))

    # Remove edges
    g_.remove_edges_from(to_remove)

    # print('here')
    # Merge edges
    mlen = len(to_merge)
    skip_index = []
    for i in range(mlen):
        e = to_merge[i]
        try:
            g_.merge_nodes(e.y,e.x)
        except:
            continue


    comps = nx.connected_components(g_)
    # # We construct an array which can map old labels to the new ones.
    # # All the labels within a connected component are assigned to a single label in the output.
    maxlab = labels1.max()

    map_array = np.arange(maxlab + 2, dtype=labels1.dtype)
    for i, nodes in enumerate(comps):
        for node in nodes:
            for label in g_.nodes[node]['labels']:
                if not label in bg_labels:
                    map_array[label] = i
                else:
                    map_array[label] = maxlab + 1
    labels2 = map_array[labels1].copy()
    return labels2

def supcls_pick_binarize(image_t): #super_map, sup_max_cls, bi_val = None): # 
    """
    pick up a certain super-pixel class or multiple classes, and binarize it into segmentation target
    Args:
        super_map:      super-pixel map
        bi_val:         if given, pick up a certain superpixel. Otherwise, draw a random one
        sup_max_cls:    max index of superpixel for avoiding overshooting when selecting superpixel

    """
    image_t = image_t.astype(np.float32)
    dst = cv2.bilateralFilter(image_t,9,75,75)
    mindst = dst.min()
    maxdst = dst.max()
    dst = dst - mindst
    dst = dst / (1e-6 + maxdst - mindst)
#     dst = (255*dst).astype('int')

    # labels1 = segmentation.slic(dst, compactness=100, n_segments=1000, start_label=0, sigma = 1.0)
    labels1 = segmentation.felzenszwalb(dst, scale = 1.0, sigma = 0.8, min_size = 400, channel_axis = None)
#     print(labels1)
    out1 = color.label2rgb(labels1, dst, kind='avg', bg_label=labels1.min())

    minout1 = out1.min()
    maxout1 = out1.max()
    out1 = out1 - minout1
    out1 = out1 / (1e-6 + maxout1 - minout1)
    out1 = (255*out1).astype('int')
    
#     plt.imshow(out1, cmap='gray')
#     plt.show()
    
    if np.unique(labels1).shape[0] == 1 and np.unique(labels1)[0] == 0:
#         print('-')
        return color.rgb2gray(out1)
    
#     try:
    g = graph.rag_mean_color(out1, labels1, mode='similarity', sigma = 127.)
#     except:
#         print(np.unique(labels1))
#         print(np.unique(out1))
    # print(g.nodes)

    labels2 = cut_thresh(g, labels1, thresh = 10)
    bg_label = labels1.max() + 1
    label_choices = [l for l in np.unique(labels2) if l != bg_label]

    if label_choices == []:
        # print('REPEAT')
        sup_max_cls = 0
        labels2 = labels1
    else:
        sup_max_cls = len(label_choices)

    out2 = color.label2rgb(labels2, dst, kind='avg', bg_label=bg_label)
    minout2 = out2.min()
    maxout2 = out2.max()
    out2 = out2 - minout2
    out2 = out2 / (1e-6 + maxout2 - minout2)
    
    out2 = (255*out2).astype('int')

    return out2[...,0]

In [11]:
# Generate pseudolabels for every image and save them
for img_fid in imgs:
# img_fid = imgs[0]

    idx = os.path.basename(img_fid).split("_")[-1].split(".nii.gz")[0]
    im_obj = sitk.ReadImage(img_fid)
    
    img = sitk.GetArrayFromImage(im_obj)
    
#     print(img.shape)
    
    out_seg = np.zeros(img.shape)
    
    for i in range(out_seg.shape[0]):
        out_seg[i] = supcls_pick_binarize(img[i])#, fg_thresh = DATASET_CONFIG[DOMAIN]['fg_thresh'] )
#         plt.imshow(out_seg[i], cmap = 'gray')
#         plt.show()

#     out_fg_o = sitk.GetImageFromArray(out_fg ) 
    out_seg_o = sitk.GetImageFromArray(out_seg )

#     out_fg_o = copy_info(im_obj, out_fg_o)
    out_seg_o = copy_info(im_obj, out_seg_o)
    seg_fid = os.path.join(out_dir, f'superpix-agun-{MODE}_{idx}.nii.gz')
#     msk_fid = os.path.join(out_dir, f'fgmask_{idx}.nii.gz')
#     sitk.WriteImage(out_fg_o, msk_fid)
    sitk.WriteImage(out_seg_o, seg_fid)
    print(f'image with id {idx} has finished')


image with id 1 has finished
image with id 2 has finished
image with id 3 has finished
image with id 5 has finished
image with id 8 has finished
image with id 10 has finished
image with id 13 has finished
image with id 15 has finished
image with id 19 has finished
image with id 20 has finished
image with id 21 has finished
image with id 22 has finished
image with id 31 has finished
image with id 32 has finished
image with id 33 has finished
image with id 34 has finished
image with id 36 has finished
image with id 37 has finished
image with id 38 has finished
image with id 39 has finished


In [9]:
MODE = 'MIDDLE' # minimum size of pesudolabels. 'MIDDLE' is the default setting

# # wrapper for process 3d image in 2d
# def superpix_vol(img, method = 'fezlen', **kwargs):
#     """
#     loop through the entire volume
#     assuming image with axis z, x, y
#     """
#     if method =='fezlen':
#         seg_func = skimage.segmentation.felzenszwalb
#     else:
#         raise NotImplementedError
        
#     out_vol = np.zeros(img.shape)
#     for ii in range(img.shape[0]):
#         if MODE == 'MIDDLE':
#             segs = seg_func(img[ii, ...], min_size = 400, sigma = 1)
#         else:
#             raise NotImplementedError
#         out_vol[ii, ...] = segs
        
#     return out_vol

# # thresholding the intensity values to get a binary mask of the patient
# def fg_mask2d(img_2d, thresh): # change this by your need
#     mask_map = np.float32(img_2d > thresh)
    
#     def getLargestCC(segmentation): # largest connected components
#         labels = label(segmentation)
#         assert( labels.max() != 0 ) # assume at least 1 CC
#         largestCC = labels == np.argmax(np.bincount(labels.flat)[1:])+1
#         return largestCC
#     if mask_map.max() < 0.999:
#         return mask_map
#     else:
#         post_mask = getLargestCC(mask_map)
#         fill_mask = snm.binary_fill_holes(post_mask)
#     return fill_mask

# # remove superpixels within the empty regions
# def superpix_masking(raw_seg2d, mask2d):
#     raw_seg2d = np.int32(raw_seg2d)
#     lbvs = np.unique(raw_seg2d)
#     max_lb = lbvs.max()
#     raw_seg2d[raw_seg2d == 0] = max_lb + 1
#     lbvs = list(lbvs)
#     lbvs.append( max_lb )
#     raw_seg2d = raw_seg2d * mask2d
#     lb_new = 1
#     out_seg2d = np.zeros(raw_seg2d.shape)
#     for lbv in lbvs:
#         if lbv == 0:
#             continue
#         else:
#             out_seg2d[raw_seg2d == lbv] = lb_new
#             lb_new += 1
    
#     return out_seg2d
            
# def superpix_wrapper(img, verbose = False, fg_thresh = 1e-4):
#     raw_seg = superpix_vol(img)
#     fg_mask_vol = np.zeros(raw_seg.shape)
#     processed_seg_vol = np.zeros(raw_seg.shape)
#     for ii in range(raw_seg.shape[0]):
#         if verbose:
#             print("doing {} slice".format(ii))
#         _fgm = fg_mask2d(img[ii, ...], fg_thresh )
#         _out_seg = superpix_masking(raw_seg[ii, ...], _fgm)
#         fg_mask_vol[ii] = _fgm
#         processed_seg_vol[ii] = _out_seg
#     return fg_mask_vol, processed_seg_vol
        
# # copy spacing and orientation info between sitk objects
def copy_info(src, dst):
    dst.SetSpacing(src.GetSpacing())
    dst.SetOrigin(src.GetOrigin())
    dst.SetDirection(src.GetDirection())
    # dst.CopyInfomation(src)
    return dst


# def strip_(img, lb):
#     img = np.int32(img)
#     if isinstance(lb, float):
#         lb = int(lb)
#         return np.float32(img == lb) * float(lb)
#     elif isinstance(lb, list):
#         out = np.zeros(img.shape)
#         for _lb in lb:
#             out += np.float32(img == int(_lb)) * float(_lb)
            
#         return out
#     else:
#         raise Exception